In [1]:
import pandas as pd

In [2]:
# Conditioning feature colums
# Or the things that we want to tell the model about our artists and their typical tracks 
c_cols = [
    # graph geometry
    "cos_sim_src_dst",
    "l2_dist_src_dst",
    
    # artist popularity
    "popularity_src",
    "popularity_dst",
    
    # neighbor features
    "mean_topk_cos_dstnbr_src",
    "mean_topk_cos_srcnbr_dst",

    # graph reachability
    "shared_neighbors",
    
]

# Y PREDICTIONS
y_cont_cols = [
    'danceability',
     
    'gender_female','gender_male',
    
    'genre_dortmund_alternative','genre_dortmund_blues','genre_dortmund_electronic',
    'genre_dortmund_folkcountry','genre_dortmund_funksoulrnb','genre_dortmund_jazz',
    'genre_dortmund_pop','genre_dortmund_raphiphop','genre_dortmund_rock',
    'genre_electronic_ambient','genre_electronic_dnb','genre_electronic_house',
    'genre_electronic_techno','genre_electronic_trance','genre_rosamerica_cla',
    'genre_rosamerica_dan','genre_rosamerica_hip','genre_rosamerica_jaz','genre_rosamerica_pop',
    'genre_rosamerica_rhy','genre_rosamerica_roc','genre_rosamerica_spe',
    'genre_tzanetakis_blu','genre_tzanetakis_cla','genre_tzanetakis_cou','genre_tzanetakis_dis',
    'genre_tzanetakis_hip','genre_tzanetakis_jaz','genre_tzanetakis_met','genre_tzanetakis_pop',
    'genre_tzanetakis_reg','genre_tzanetakis_roc',
    
    'ismir04_rhythm_jive','ismir04_rhythm_quickstep',
    'ismir04_rhythm_rumba_american','ismir04_rhythm_rumba_international','ismir04_rhythm_rumba_misc',
    'ismir04_rhythm_samba','ismir04_rhythm_tango','ismir04_rhythm_waltz',
    # SKIPPING THESE BECAUSE THE MODEL SKIPPED THEM PREVIOUSLY
    # 'ismir04_rhythm_viennesewaltz','ismir04_rhythm_chachacha', 
    
    'mood_acoustic','mood_aggressive','mood_electronic','mood_happy',
    'mood_party','mood_relaxed','mood_sad',
    
    'moods_mirex_cluster1','moods_mirex_cluster2',
    'moods_mirex_cluster3','moods_mirex_cluster4','moods_mirex_cluster5',
    
    'timbre_bright','timbre_dark',
    
    'tonal_atonal_atonal','tonal_atonal_tonal',
    
    'voice_instrumental_instrumental','voice_instrumental_voice'
]
y_bin_cols = []


## Get the track embeddings and features

In [3]:
# First read in the model's sample track features for training
# Then get those track features' artist embeddings.
# Join them
# Train on a Linear Regression Model 

In [4]:
ARTIST_EMBEDDINGS_X = '/tmp/artist_embeddings_collab_neg.csv'
artist_embeddings = pd.read_csv(ARTIST_EMBEDDINGS_X)

parquet_path = "block_track_features.parquet"
block_track_features = pd.read_parquet(parquet_path)

In [5]:
artist_embeddings.head(2)

,cos_sim_src_dst,l2_dist_src_dst,dot_src_dst,abs_diff_mean,abs_diff_max,popularity_src,popularity_dst,max_cos_srcnbr_dst,mean_cos_srcnbr_dst,mean_topk_cos_srcnbr_dst,...,mean_cos_dstnbr_src,mean_topk_cos_dstnbr_src,num_dst_neighbors,shared_neighbors,jaccard_neighbors,adamic_adar,preferential_attachment,src,dst,label
0,0.836101,0.572536,0.836101,0.086337,0.224102,2.1180,2.5617,0.853362,0.621910,0.621910,...,0.927103,0.927103,3,1,0.20,0.721348,9,750619,856447,1
1,0.776645,0.668364,0.776645,0.099268,0.215586,3.0418,1.9727,0.904819,0.900058,0.900058,...,0.836941,0.836941,2,1,0.25,0.721348,6,662124,847889,1


In [6]:
block_track_features.head(2)

,src,dst,recording_gid,recording_id,danceability,gender_female,gender_male,genre_dortmund_alternative,genre_dortmund_blues,genre_dortmund_electronic,...,moods_mirex_cluster2,moods_mirex_cluster3,moods_mirex_cluster4,moods_mirex_cluster5,timbre_bright,timbre_dark,tonal_atonal_atonal,tonal_atonal_tonal,voice_instrumental_instrumental,voice_instrumental_voice
0,277063,418369,aef1d4b3-b2e7-462f-be5d-5df774293fe5,14644528,0.979156,0.814088,0.185912,2.170894e-14,2.817410e-14,1.000000,...,0.044781,0.227540,0.030868,0.619896,0.029254,0.970746,0.857114,0.142886,0.999995,0.000005
1,402608,504713,36c73484-be5f-4f4f-bab0-9d05a2aa7d50,7450506,0.960337,0.010670,0.989330,3.314676e-01,1.661954e-01,0.047832,...,0.067013,0.746356,0.061295,0.028264,0.547201,0.452799,0.002137,0.997863,0.992258,0.007742


In [7]:
artist_embeddings_unique = (
    artist_embeddings
        .groupby(['src','dst'], as_index=False)
        .mean(numeric_only=True)
)


In [8]:
df = block_track_features.merge(
    artist_embeddings_unique,
    on=['src','dst'],
    how='left',
    validate='many_to_one'   # <-- left repeats allowed, right must be unique
)

In [9]:
sample_size = int(round(len(df) * 0.85, 0))
sample_size_pair = 50_000 # Reduced sample size 
val_size = 10_000

df_train = df.sample(n=sample_size, axis="index", random_state=42)
df_val = df.loc[~df.index.isin(df_train.index)]

df_train = df_train.sample(n=sample_size_pair, random_state=67)
df_val = df_val.sample(n=val_size,random_state=67)



In [10]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

X_cols = y_cont_cols + y_bin_cols + c_cols # All of the inputs to the CVAE model we are using for the popularity model too
y_col = "track_popularity"

def fit_popularity_linear_regression(df_train, X_cols, y_col):
    X = df_train[X_cols].values
    y = df_train[y_col].values

    model = LinearRegression()
    model.fit(X, y)

    return model

def eval_popularity_model(model, df, X_cols, y_col):
    X = df[X_cols].values
    y = df[y_col].values

    y_hat = model.predict(X)

    return {
        "r2": r2_score(y, y_hat),
        "rmse": mean_absolute_error(y, y_hat, squared=False),
    }


In [11]:
## Take these and predict the new tracks

pop_model = fit_popularity_linear_regression(df_train, X_cols, y_col)

train_metrics = eval_popularity_model(pop_model, df_train, X_cols, y_col)
val_metrics = eval_popularity_model(pop_model, df_val, X_cols, y_col)

print("Train:", train_metrics)
print("Val:", val_metrics)


KeyError: 'track_popularity'

In [ ]:
coef_df = (
    pd.DataFrame({
        "feature": X_cols,
        "coef": pop_model.coef_,
    })
    .sort_values("coef", key=np.abs, ascending=False)
)

coef_df


In [ ]:
def score_generated_tracks(pop_model, yc_samples, yb_probs=None):
    n_rows, n_samples, y_dim = yc_samples.shape

    if yb_probs is not None:
        X = np.concatenate(
            [yc_samples, yb_probs],
            axis=-1
        )
    else:
        X = yc_samples

    scores = pop_model.predict(X.reshape(-1, X.shape[-1]))
    return scores.reshape(n_rows, n_samples)


In [ ]:
scores = score_generated_tracks(pop_model, df_val[y_cont_cols], df_val[y_bin_cols])
best_score = scores.max(axis=1)  # best synthetic track per conditioning row